# Zebra finch calls: layer sweep and full test re-run

This notebook adds two analyses to the bird-disjoint benchmark.

**3a. Layer sweep.** Every transformer layer of the four frozen encoders is mean-pooled and scored with logistic regression on the validation birds only. The best validation layer is fixed before the test split is touched.

**4. Full test re-run.** All four heads (logistic regression, MLP, XGBoost, BiLSTM) are trained for every encoder at 120 clips per class over seeds 42 to 46. This is done for the final layer (the original protocol) and for the selected layer. Every run saves clip-level predictions with the bird ID, so bird-level bootstrap intervals and leave-one-bird-out checks can be reported for every system.

The settings, checkpoints, model revisions and weight hashes are the same as in the original notebook. The final-layer re-run is checked against the archived numbers at the end.

**How to run on Kaggle**
1. Upload `zf_kaggle_data.zip` as a new Kaggle dataset and add it to this notebook (Add input).
2. Settings: Accelerator **GPU T4 x1** (or P100), Internet **on** (the models are downloaded from Hugging Face and Google Cloud Storage).
3. Run all cells. Expect about 1 to 1.5 hours. If the session stops, run all again: finished encoders and seeds are skipped.
4. Download `/kaggle/working/zf_layer_test_results.zip` and send it back.

On Colab, unzip the data under `/content` and use a GPU runtime. The paths are found automatically.

## 1. Setup and data checks
The cell stops if a bird appears in two splits, if an audio file is missing, or if a training subset is not nested, balanced and drawn from the training birds.

In [ ]:
# ============================================================
# Cell 1. Setup, paths and data integrity checks
# ============================================================
import os, sys, json, time, random, hashlib, warnings, subprocess, gc, shutil, platform
from pathlib import Path
import numpy as np
import pandas as pd

try:
    import xgboost
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost"], check=True)
    import xgboost
import sklearn
import torch

# ---------- paths ----------
def find_data_dir():
    """Find the folder that holds processed_manifest.csv, audio_16k/ and subsets/."""
    if os.environ.get("ZF_DATA_DIR"):
        return Path(os.environ["ZF_DATA_DIR"])
    for root in [Path("/kaggle/input"), Path("/content"), Path.cwd()]:
        if not root.exists():
            continue
        for depth in range(0, 5):
            hits = sorted(root.glob("/".join(["*"] * depth + ["processed_manifest.csv"])))
            hits = [h for h in hits if (h.parent / "audio_16k").is_dir()]
            if hits:
                return hits[0].parent
    raise FileNotFoundError(
        "processed_manifest.csv was not found. Add the zf_kaggle_data dataset "
        "(or unzip it under /content) and run this cell again.")

DATA_DIR = find_data_dir()
AUDIO_DIR = DATA_DIR / "audio_16k"
SUBSET_DIR = DATA_DIR / "subsets"
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content")
OUT_DIR = Path(os.environ.get("ZF_OUT_DIR", WORK_ROOT / "zf_layer_test_v1"))
# the feature cache (about 4-5 GB) lives outside /kaggle/working so it is not saved as notebook output
CACHE_DIR = Path(os.environ.get("ZF_CACHE_DIR", "/tmp/zf_feature_cache_v1" if WORK_ROOT.name == "working"
                                else WORK_ROOT / "zf_feature_cache_v1"))
MODEL_DIR = WORK_ROOT / "zf_models"
for d in [OUT_DIR, CACHE_DIR, MODEL_DIR, OUT_DIR / "figures"]:
    d.mkdir(parents=True, exist_ok=True)

# ---------- fixed protocol (same as the original study) ----------
SEEDS = [42, 43, 44, 45, 46]
BUDGET = 120                       # clips per class used for the layer sweep and the test re-run
ALL_BUDGETS = [50, 100, 120]       # only used to verify that the subsets are nested
LABELS = ["Aggressive", "Distance", "Nest", "Tet", "Thuk", "Tuck"]
LAB2ID = {l: i for i, l in enumerate(LABELS)}
C_GRID = [0.01, 0.1, 1.0]
N_BOOT = 1000

# ---------- manifest checks ----------
manifest = pd.read_csv(DATA_DIR / "processed_manifest.csv", dtype={"clip_id": str, "group_id": str})
manifest = manifest.sort_values("clip_id").reset_index(drop=True)
assert manifest.clip_id.is_unique, "duplicate clip_id in manifest"
assert set(manifest.final_label) == set(LABELS), sorted(set(manifest.final_label))
assert set(manifest.split) == {"train", "validation", "test"}
birds = {s: set(manifest.loc[manifest.split == s, "group_id"]) for s in ["train", "validation", "test"]}
assert not (birds["train"] & birds["validation"]) and not (birds["train"] & birds["test"]) \
    and not (birds["validation"] & birds["test"]), "a bird appears in more than one split"
missing = [c for c in manifest.clip_id if not (AUDIO_DIR / f"{c}.wav").is_file()]
assert not missing, f"{len(missing)} audio files are missing, e.g. {missing[:3]}"

train_ids = set(manifest.loc[manifest.split == "train", "clip_id"])
label_of = manifest.set_index("clip_id")["final_label"]
bird_of = manifest.set_index("clip_id")["group_id"]
val_df = manifest[manifest.split == "validation"].reset_index(drop=True)
test_df = manifest[manifest.split == "test"].reset_index(drop=True)
y_val = val_df.final_label.map(LAB2ID).to_numpy()
y_test = test_df.final_label.map(LAB2ID).to_numpy()

# ---------- seeded subset checks ----------
subsets = {}
for seed in SEEDS:
    previous = set()
    for b in ALL_BUDGETS:
        sub = pd.read_csv(SUBSET_DIR / f"train_seed_{seed}_budget_{b}.csv", dtype={"clip_id": str})
        sub = sub.sort_values("clip_id").reset_index(drop=True)
        ids = set(sub.clip_id)
        assert sub.clip_id.is_unique and len(sub) == b * len(LABELS)
        assert ids <= train_ids, "subset contains non-training clips"
        assert previous <= ids, "subsets are not nested"
        assert (sub.final_label.to_numpy() == sub.clip_id.map(label_of).to_numpy()).all()
        assert (sub.final_label.value_counts().reindex(LABELS) == b).all()
        previous = ids
        if b == BUDGET:
            subsets[seed] = sub[["clip_id", "final_label"]].assign(group_id=sub.clip_id.map(bird_of))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_JOBS = max(1, min(4, os.cpu_count() or 1))

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

print("Data folder :", DATA_DIR)
print("Output      :", OUT_DIR)
print("Device      :", DEVICE, "| CPU workers:", N_JOBS)
print("Clips       :", manifest.split.value_counts().to_dict())
print("Birds       :", {k: len(v) for k, v in birds.items()})
print("Versions    : torch", torch.__version__, "| sklearn", sklearn.__version__, "| xgboost", xgboost.__version__)
print("All data checks passed.")

## 2. Encoders
HuBERT and wav2vec 2.0 are pinned to the exact Hugging Face revisions of the original run. The BirdAVES weights must match the original SHA-256 hashes. All encoders stay frozen.

In [ ]:
# ============================================================
# Cell 2. Encoder definitions (same checkpoints as the original study)
# ============================================================
import torchaudio
from urllib.request import urlopen
from transformers import AutoFeatureExtractor, HubertModel, Wav2Vec2Model
assert hasattr(torchaudio, "models") and hasattr(torchaudio.models, "wav2vec2_model"), (
    "This torchaudio build has no wav2vec2_model. Run: pip install -q 'torchaudio<2.9' "
    "matching your torch version, then restart the session.")

GCS = "https://storage.googleapis.com/esp-public-files/birdaves/"
ENCODERS = {
    "HuBERT": dict(kind="hf", model_id="facebook/hubert-base-ls960",
                   revision="dba3bb02fda4248b6e082697eee756de8fe8aa8a"),
    "Wav2Vec2": dict(kind="hf", model_id="facebook/wav2vec2-base",
                     revision="0b5b8e868dd84f03fd87d01f9c4ff0f080fecfe8"),
    "BirdAVES-base": dict(kind="ta", weights=GCS + "birdaves-biox-base.torchaudio.pt",
                          config=GCS + "birdaves-biox-base.torchaudio.model_config.json",
                          sha256="3427869cb939e48fa3052db0f2e611705b83f42e8f9feab65280d9d3f195d72a"),
    "BirdAVES-large": dict(kind="ta", weights=GCS + "birdaves-biox-large.torchaudio.pt",
                           config=GCS + "birdaves-biox-large.torchaudio.model_config.json",
                           sha256="63050b721c103a37cd2fba91c7b744b80075b819f02dbc867344138e2a92c5bf"),
}
ENCODER_NAMES = list(ENCODERS)


def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def download(url, dest):
    if dest.is_file():
        return
    tmp = dest.with_suffix(dest.suffix + ".part")
    with urlopen(url, timeout=300) as r, open(tmp, "wb") as f:
        shutil.copyfileobj(r, f)
    tmp.replace(dest)


def load_encoder(name):
    """Return (run, model, n_layers, record). run(audio) gives a list of n_layers arrays of shape (T, D):
    the output of every transformer layer, layer 1 first and the final layer last."""
    spec = ENCODERS[name]
    record = {"encoder": name, **spec, "torch": torch.__version__, "torchaudio": torchaudio.__version__}
    if spec["kind"] == "hf":
        import transformers
        record["transformers"] = transformers.__version__
        cls = HubertModel if "hubert" in spec["model_id"] else Wav2Vec2Model
        proc = AutoFeatureExtractor.from_pretrained(spec["model_id"], revision=spec["revision"])
        model = cls.from_pretrained(spec["model_id"], revision=spec["revision"]).to(DEVICE).eval()
        model.requires_grad_(False)
        n_layers = model.config.num_hidden_layers
        record["input_normalization"] = bool(proc.do_normalize)

        def run(audio):
            inputs = proc(audio, sampling_rate=16000, return_tensors="pt", padding=False)
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            with torch.inference_mode():
                out = model(**inputs, output_hidden_states=True)
            hs = out.hidden_states[1:]           # drop the pre-transformer embedding output
            # the final entry is exactly the last_hidden_state used in the original study
            assert torch.allclose(hs[-1], out.last_hidden_state)
            return [h[0].float().cpu().numpy() for h in hs]
    else:
        d = MODEL_DIR / name
        d.mkdir(parents=True, exist_ok=True)
        w, c = d / "model.pt", d / "model_config.json"
        download(spec["weights"], w)
        download(spec["config"], c)
        digest = file_sha256(w)
        assert digest == spec["sha256"], f"{name}: weight hash {digest} differs from the original study"
        config = json.loads(c.read_text())
        model = torchaudio.models.wav2vec2_model(**config, aux_num_out=None)
        ckpt = torch.load(w, map_location="cpu", weights_only=True)
        if isinstance(ckpt, dict) and "state_dict" in ckpt:
            ckpt = ckpt["state_dict"]
        elif isinstance(ckpt, dict) and "model" in ckpt:
            ckpt = ckpt["model"]
        model.load_state_dict(ckpt, strict=True)
        del ckpt
        model = model.to(DEVICE).eval()
        model.requires_grad_(False)
        n_layers = config["encoder_num_layers"]
        record["input_normalization"] = "zero mean, unit variance (eps 1e-7)"

        def run(audio):
            x = ((audio - audio.mean()) / np.sqrt(audio.var() + 1e-7)).astype(np.float32)
            x = torch.from_numpy(x).unsqueeze(0).to(DEVICE)
            with torch.inference_mode():
                feats, _ = model.extract_features(x)   # one tensor per transformer layer
            return [f[0].float().cpu().numpy() for f in feats]

    record["n_layers"] = int(n_layers)
    return run, model, int(n_layers), record

print("Encoders:", ENCODER_NAMES)

## 3. Feature extraction
Each clip passes through each encoder once. All transformer layers are kept, both mean-pooled and as frame sequences. The final layer equals the `last_hidden_state` used originally, and the code asserts this.

In [ ]:
# ============================================================
# Cell 3. Extract every transformer layer once and cache it
#   pooled.npy     (clips, layers, dim)   mean over frames, float32
#   sequences.npy  (frames, layers, dim)  frame sequences for the BiLSTM, float32
#   offsets.npy    clip i owns frames offsets[i]:offsets[i+1]
# A finished encoder is skipped, so the cell can be re-run after a restart.
# ============================================================
import soundfile as sf
from tqdm.auto import tqdm

CLIP_IDS = manifest.clip_id.tolist()


def extract(name):
    d = CACHE_DIR / name
    if (d / "done.json").is_file():
        print(f"{name}: cache found, skipping")
        return
    d.mkdir(parents=True, exist_ok=True)
    run, model, n_layers, record = load_encoder(name)
    pooled, seqs, lengths = None, [], []
    t0 = time.time()
    for i, cid in enumerate(tqdm(CLIP_IDS, desc=name)):
        audio, sr = sf.read(AUDIO_DIR / f"{cid}.wav", dtype="float32")
        assert sr == 16000 and audio.ndim == 1 and len(audio) > 0 and np.isfinite(audio).all(), cid
        layers = run(audio)
        assert len(layers) == n_layers, (name, len(layers))
        stack = np.stack(layers).astype(np.float32)          # (L, T, D)
        assert stack.shape[1] > 0 and np.isfinite(stack).all(), cid
        if pooled is None:
            pooled = np.zeros((len(CLIP_IDS), n_layers, stack.shape[2]), np.float32)
        pooled[i] = stack.mean(axis=1)
        seqs.append(np.ascontiguousarray(stack.transpose(1, 0, 2)))   # (T, L, D)
        lengths.append(stack.shape[1])
    offsets = np.concatenate([[0], np.cumsum(lengths)]).astype(np.int64)
    np.save(d / "pooled.npy", pooled)
    np.save(d / "sequences.npy", np.concatenate(seqs, axis=0))
    np.save(d / "offsets.npy", offsets)
    (d / "clip_ids.txt").write_text("\n".join(CLIP_IDS))
    record.update(dim=int(pooled.shape[2]), clips=len(CLIP_IDS), frames=int(offsets[-1]),
                  min_frames=int(min(lengths)), seconds=round(time.time() - t0, 1),
                  device=str(DEVICE), gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
    (d / "done.json").write_text(json.dumps(record, indent=2))
    print(json.dumps({k: record[k] for k in ["n_layers", "dim", "frames", "min_frames", "seconds"]}))
    del run, model, pooled, seqs
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


for name in ENCODER_NAMES:
    extract(name)

records = {n: json.loads((CACHE_DIR / n / "done.json").read_text()) for n in ENCODER_NAMES}
(OUT_DIR / "encoder_records.json").write_text(json.dumps(records, indent=2))
print("Feature extraction finished.")

## 4. Pooled-vector classifiers

In [ ]:
# ============================================================
# Cell 4. Feature access and the three pooled-vector classifiers
#   settings are copied from the original notebook cells 67-69
# ============================================================
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, accuracy_score
from sklearn.exceptions import ConvergenceWarning
from xgboost import XGBClassifier
from joblib import Parallel, delayed


class Cache:
    def __init__(self, name):
        d = CACHE_DIR / name
        self.name = name
        self.meta = json.loads((d / "done.json").read_text())
        self.pooled = np.load(d / "pooled.npy")
        self.seq = np.load(d / "sequences.npy", mmap_mode="r")
        self.off = np.load(d / "offsets.npy")
        ids = (d / "clip_ids.txt").read_text().split("\n")
        self.pos = {c: i for i, c in enumerate(ids)}
        self.n_layers = self.pooled.shape[1]

    def X(self, clip_ids, layer):
        """Mean-pooled vectors of transformer layer `layer` (1 = first, n_layers = final)."""
        return self.pooled[[self.pos[c] for c in clip_ids], layer - 1]

    def sequences(self, clip_ids, layer):
        out = []
        for c in clip_ids:
            i = self.pos[c]
            out.append(np.asarray(self.seq[self.off[i]:self.off[i + 1], layer - 1], dtype=np.float32))
        return out


def macro_f1(y, p):
    return float(f1_score(y, p, labels=list(range(len(LABELS))), average="macro", zero_division=0))


def fit_logreg(Xtr, ytr, Xv, yv):
    """C chosen on validation macro-F1 only (ties keep the smaller C, as in the original)."""
    best = None
    for C in C_GRID:
        m = make_pipeline(StandardScaler(),
                          LogisticRegression(C=C, class_weight="balanced", solver="lbfgs", max_iter=5000))
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            m.fit(Xtr, ytr)
        pv = m.predict(Xv)
        f = macro_f1(yv, pv)
        if best is None or f > best["val_f1"]:
            best = dict(val_f1=f, C=C, model=m, pv=pv,
                        converged=not any(issubclass(w.category, ConvergenceWarning) for w in caught))
    return best


def fit_mlp(Xtr, ytr, seed):
    m = make_pipeline(StandardScaler(),
                      MLPClassifier(hidden_layer_sizes=(128,), activation="relu", solver="adam", alpha=1e-4,
                                    batch_size=32, learning_rate_init=1e-3, max_iter=300,
                                    early_stopping=False, random_state=seed))
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        m.fit(Xtr, ytr)
    return m, not any(issubclass(w.category, ConvergenceWarning) for w in caught)


def fit_xgb(Xtr, ytr, seed):
    m = XGBClassifier(objective="multi:softprob", num_class=len(LABELS), random_state=seed, verbosity=0,
                      n_estimators=150, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                      min_child_weight=1, reg_lambda=1.0, tree_method="hist", eval_metric="mlogloss", n_jobs=2)
    m.fit(Xtr, ytr)          # no scaling, as in the original
    return m


VAL_IDS = val_df.clip_id.tolist()
TEST_IDS = test_df.clip_id.tolist()
TRAIN = {s: (subsets[s].clip_id.tolist(), subsets[s].final_label.map(LAB2ID).to_numpy()) for s in SEEDS}
print("Heads ready. Training clips per seed:", {s: len(v[0]) for s, v in TRAIN.items()})

## 5. BiLSTM on frame sequences

In [ ]:
# ============================================================
# Cell 5. BiLSTM on frame sequences (copied from original notebook cell 71)
#   64 units each way, dropout 0.3, AdamW 1e-3, weight decay 1e-4, batch 32,
#   up to 30 epochs, patience 6 on validation macro-F1, gradient clip 1.0,
#   per-dimension standardization fitted on training frames (std floor 1e-6)
# ============================================================
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence

MAX_EPOCHS, PATIENCE, BATCH_SIZE, HIDDEN_SIZE, LR_RATE, WEIGHT_DECAY = 30, 6, 32, 64, 1e-3, 1e-4


class SequenceDataset(Dataset):
    def __init__(self, sequences, labels, mean, std):
        self.labels = np.asarray(labels, dtype=np.int64)
        self.sequences = [torch.tensor(np.nan_to_num(((s - mean) / std).astype(np.float32),
                                                     nan=0.0, posinf=0.0, neginf=0.0)) for s in sequences]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return self.sequences[i], int(self.labels[i])


def collate_sequences(batch):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([s.shape[0] for s in seqs], dtype=torch.long)
    return pad_sequence(seqs, batch_first=True, padding_value=0.0), lengths, torch.tensor(labels, dtype=torch.long)


class BiLSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_size, num_classes):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_size, num_layers=1,
                            batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.30)
        self.classifier = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x, lengths):
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hidden, _) = self.lstm(packed)
        return self.classifier(self.dropout(torch.cat([hidden[-2], hidden[-1]], dim=1)))


@torch.no_grad()
def predict_loader(model, loader):
    model.eval()
    out = []
    for x, lengths, _ in loader:
        out.extend(model(x.to(DEVICE), lengths.to(DEVICE)).argmax(dim=1).cpu().numpy().tolist())
    return np.asarray(out)


def fit_bilstm(train_seqs, ytr, val_seqs, yv, test_seqs, yt, seed):
    set_seed(seed)
    frames = np.concatenate(train_seqs, axis=0).astype(np.float32)
    mean = frames.mean(axis=0).astype(np.float32)
    std = frames.std(axis=0).astype(np.float32)
    std[std < 1e-6] = 1.0
    mk = lambda s, y, sh: DataLoader(SequenceDataset(s, y, mean, std), batch_size=BATCH_SIZE, shuffle=sh,
                                     collate_fn=collate_sequences, num_workers=0)
    train_loader, val_loader, test_loader = mk(train_seqs, ytr, True), mk(val_seqs, yv, False), mk(test_seqs, yt, False)
    model = BiLSTMClassifier(train_seqs[0].shape[1], HIDDEN_SIZE, len(LABELS)).to(DEVICE)
    counts = np.bincount(ytr, minlength=len(LABELS)).astype(np.float32)
    weights = np.zeros(len(LABELS), np.float32)
    weights[counts > 0] = len(ytr) / (len(LABELS) * counts[counts > 0])
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, device=DEVICE))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR_RATE, weight_decay=WEIGHT_DECAY)
    best_f1, best_epoch, best_state, stale = -1.0, 0, None, 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for x, lengths, y in train_loader:
            x, lengths, y = x.to(DEVICE), lengths.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x, lengths), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        f = macro_f1(yv, predict_loader(model, val_loader))
        if f > best_f1 + 1e-5:
            best_f1, best_epoch, stale = f, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
        if stale >= PATIENCE:
            break
    model.load_state_dict(best_state)
    pv, pt = predict_loader(model, val_loader), predict_loader(model, test_loader)
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return pv, pt, best_epoch

print("BiLSTM ready on", DEVICE)

## 6. Layer sweep (validation only)

In [ ]:
# ============================================================
# Cell 6. Layer sweep on VALIDATION only (logistic regression, 120 clips/class, 5 seeds)
#   The test split is not touched here. The selected layer is the one with the
#   highest mean validation macro-F1 across seeds; ties go to the deeper layer.
# ============================================================

def sweep_one(name, layer, Xall_val, train_X):
    rows = []
    for seed in SEEDS:
        ids, ytr = TRAIN[seed]
        r = fit_logreg(train_X[seed], ytr, Xall_val, y_val)
        rows.append(dict(encoder=name, layer=layer, seed=seed, budget=BUDGET, C=r["C"],
                         val_macro_f1=r["val_f1"], val_accuracy=float(accuracy_score(y_val, r["pv"])),
                         converged=r["converged"]))
    return rows


sweep_path = OUT_DIR / "layer_sweep_runs.csv"
sweep = pd.read_csv(sweep_path) if sweep_path.is_file() else pd.DataFrame()
for name in ENCODER_NAMES:
    if len(sweep) and (sweep.encoder == name).any():
        print(f"{name}: sweep already done")
        continue
    cache = Cache(name)
    t0 = time.time()
    jobs = []
    for layer in range(1, cache.n_layers + 1):
        Xv = cache.X(VAL_IDS, layer)
        tx = {s: cache.X(TRAIN[s][0], layer) for s in SEEDS}
        jobs.append(delayed(sweep_one)(name, layer, Xv, tx))
    out = Parallel(n_jobs=N_JOBS)(jobs)
    sweep = pd.concat([sweep, pd.DataFrame([r for rows in out for r in rows])], ignore_index=True)
    sweep.to_csv(sweep_path, index=False)
    print(f"{name}: {cache.n_layers} layers swept in {time.time() - t0:.0f} s")
    del cache
    gc.collect()

sweep_summary = (sweep.groupby(["encoder", "layer"])
                 .agg(val_macro_f1=("val_macro_f1", "mean"), val_macro_f1_sd=("val_macro_f1", "std"),
                      val_accuracy=("val_accuracy", "mean"), not_converged=("converged", lambda c: int((~c.astype(bool)).sum())))
                 .reset_index())
n_layers_of = {n: int(sweep.loc[sweep.encoder == n, "layer"].max()) for n in ENCODER_NAMES}
sweep_summary["n_layers"] = sweep_summary.encoder.map(n_layers_of)
sweep_summary["relative_depth"] = sweep_summary.layer / sweep_summary.n_layers
sweep_summary.to_csv(OUT_DIR / "layer_sweep_summary.csv", index=False)

SELECTED = {}
for name in ENCODER_NAMES:
    s = sweep_summary[sweep_summary.encoder == name]
    top = s.val_macro_f1.max()
    best_layer = int(s.loc[np.isclose(s.val_macro_f1, top, rtol=0, atol=1e-12), "layer"].max())
    final = n_layers_of[name]
    SELECTED[name] = dict(best_layer=best_layer, final_layer=final,
                          best_val_macro_f1=float(top),
                          final_val_macro_f1=float(s.loc[s.layer == final, "val_macro_f1"].iloc[0]))
(OUT_DIR / "selected_layers.json").write_text(json.dumps(SELECTED, indent=2))
print(pd.DataFrame(SELECTED).T.to_string())

## 7. Final test re-run with clip-level predictions

In [ ]:
# ============================================================
# Cell 7. Final test: 4 encoders x {final layer, best validation layer} x 4 heads x 5 seeds
#   Every run stores clip-level predictions for validation and test, so bird-level
#   bootstrap intervals and leave-one-bird-out checks can be computed for every system.
#   Progress is saved after each (encoder, layer, seed); re-running the cell resumes.
# ============================================================
HEADS = ["logreg", "mlp", "xgboost", "bilstm"]
runs_path, preds_path = OUT_DIR / "test_runs.csv", OUT_DIR / "test_predictions.csv"
runs = pd.read_csv(runs_path) if runs_path.is_file() else pd.DataFrame()
done = set(map(tuple, runs[["encoder", "layer", "seed"]].drop_duplicates().to_numpy().tolist())) if len(runs) else set()


def pred_frame(name, layer, head, seed, split, ids, y, p):
    return pd.DataFrame(dict(encoder=name, layer=layer, head=head, seed=seed, budget=BUDGET, split=split,
                             clip_id=ids, group_id=[bird_of[c] for c in ids],
                             true=[LABELS[i] for i in y], pred=[LABELS[i] for i in p]))


for name in ENCODER_NAMES:
    cache = Cache(name)
    sel = SELECTED[name]
    for layer in sorted({sel["final_layer"], sel["best_layer"]}):
        Xv, Xt = cache.X(VAL_IDS, layer), cache.X(TEST_IDS, layer)
        Sv, St = cache.sequences(VAL_IDS, layer), cache.sequences(TEST_IDS, layer)
        for seed in SEEDS:
            if (name, layer, seed) in done:
                continue
            ids, ytr = TRAIN[seed]
            Xtr, Str = cache.X(ids, layer), cache.sequences(ids, layer)
            new_rows, new_preds = [], []
            t0 = time.time()
            for head in HEADS:
                extra = {}
                if head == "logreg":
                    r = fit_logreg(Xtr, ytr, Xv, y_val)
                    pv, pt = r["pv"], r["model"].predict(Xt)
                    extra = dict(C=r["C"], converged=r["converged"])
                elif head == "mlp":
                    m, conv = fit_mlp(Xtr, ytr, seed)
                    pv, pt = m.predict(Xv), m.predict(Xt)
                    extra = dict(converged=conv)
                elif head == "xgboost":
                    m = fit_xgb(Xtr, ytr, seed)
                    pv, pt = m.predict(Xv), m.predict(Xt)
                else:
                    pv, pt, ep = fit_bilstm(Str, ytr, Sv, y_val, St, y_test, seed)
                    extra = dict(best_epoch=ep)
                pv, pt = np.asarray(pv).astype(int), np.asarray(pt).astype(int)
                new_rows.append(dict(encoder=name, layer=layer, n_layers=sel["final_layer"],
                                     is_final_layer=layer == sel["final_layer"],
                                     is_best_val_layer=layer == sel["best_layer"],
                                     head=head, seed=seed, budget=BUDGET,
                                     val_macro_f1=macro_f1(y_val, pv), val_accuracy=float(accuracy_score(y_val, pv)),
                                     test_macro_f1=macro_f1(y_test, pt), test_accuracy=float(accuracy_score(y_test, pt)),
                                     **extra))
                new_preds += [pred_frame(name, layer, head, seed, "validation", VAL_IDS, y_val, pv),
                              pred_frame(name, layer, head, seed, "test", TEST_IDS, y_test, pt)]
            # predictions first, run log second: a crash in between only leaves duplicates,
            # which the summary cell drops
            pd.concat(new_preds).to_csv(preds_path, mode="a", header=not preds_path.is_file(), index=False)
            runs = pd.concat([runs, pd.DataFrame(new_rows)], ignore_index=True)
            runs.to_csv(runs_path, index=False)
            done.add((name, layer, seed))
            msg = " ".join(f"{r['head']}={r['test_macro_f1']:.3f}" for r in new_rows)
            print(f"{name} L{layer:>2} seed {seed}: {msg}  ({time.time() - t0:.0f} s)")
    del cache
    gc.collect()

print("Test runs complete:", len(runs), "rows")

## 8. Summary with bird-level uncertainty and reproduction check
`REPRODUCED` means the final-layer re-run is within 0.01 macro-F1 of the archived value (0.03 for the BiLSTM, whose GPU kernels are not fully deterministic).

In [ ]:
# ============================================================
# Cell 8. Test summary with bird-level uncertainty
#   macro-F1 is averaged over the five seeds; the 95% interval resamples the six
#   test birds with replacement (1000 draws); LOBO drops one test bird at a time.
#   If baseline_predictions.csv (MFCC features) is in the data folder, paired
#   bird-bootstrap differences against it are added.
# ============================================================
preds = pd.read_csv(preds_path, dtype={"clip_id": str, "group_id": str})
preds = preds.drop_duplicates(["encoder", "layer", "head", "seed", "split", "clip_id"], keep="last")
runs = pd.read_csv(runs_path).drop_duplicates(["encoder", "layer", "head", "seed"], keep="last")
TEST_BIRDS = sorted(test_df.group_id.unique())
rng = np.random.default_rng(1)
BOOT = [rng.choice(TEST_BIRDS, len(TEST_BIRDS), replace=True) for _ in range(N_BOOT)]


def seed_mean_f1(d):
    return float(np.mean([f1_score(x.true, x.pred, labels=LABELS, average="macro", zero_division=0)
                          for _, x in d.groupby("seed")]))


def by_bird(d):
    return {b: d[d.group_id == b] for b in TEST_BIRDS}


def boot_scores(g):
    return np.array([seed_mean_f1(pd.concat([g[b] for b in draw])) for draw in BOOT])


test_preds = preds[preds.split == "test"]
base_path = DATA_DIR / "baseline_predictions.csv"
baseline = {}
if base_path.is_file():
    bp = pd.read_csv(base_path, dtype={"clip_id": str, "group_id": str})
    bp = bp[(bp.split == "test") & (bp.budget == BUDGET) & (bp.features == "acoustic")]
    for h in ["logreg", "gboost"]:
        g = by_bird(bp[bp["head"] == h])
        baseline[h] = dict(f1=seed_mean_f1(bp[bp["head"] == h]), boot=boot_scores(g))
    print("MFCC baseline test macro-F1:", {h: round(v["f1"], 4) for h, v in baseline.items()})

rows = []
for (name, layer, head), d in test_preds.groupby(["encoder", "layer", "head"]):
    g = by_bird(d)
    bs = boot_scores(g)
    lobo = [seed_mean_f1(d[d.group_id != b]) for b in TEST_BIRDS]
    r = runs[(runs.encoder == name) & (runs.layer == layer) & (runs["head"] == head)]
    row = dict(encoder=name, layer=int(layer), n_layers=int(r.n_layers.iloc[0]),
               is_final_layer=bool(r.is_final_layer.iloc[0]), is_best_val_layer=bool(r.is_best_val_layer.iloc[0]),
               head=head, runs=len(r),
               val_macro_f1=r.val_macro_f1.mean(), val_macro_f1_sd=r.val_macro_f1.std(),
               test_macro_f1=seed_mean_f1(d), test_macro_f1_seed_sd=r.test_macro_f1.std(),
               test_accuracy=r.test_accuracy.mean(),
               bird_boot_lo=float(np.percentile(bs, 2.5)), bird_boot_hi=float(np.percentile(bs, 97.5)),
               lobo_min=min(lobo), lobo_max=max(lobo))
    for h, b in baseline.items():
        diff = bs - b["boot"]                     # same resampled birds for both systems
        row[f"minus_mfcc_{h}"] = row["test_macro_f1"] - b["f1"]
        row[f"minus_mfcc_{h}_lo"] = float(np.percentile(diff, 2.5))
        row[f"minus_mfcc_{h}_hi"] = float(np.percentile(diff, 97.5))
    rows.append(row)
summary = pd.DataFrame(rows).sort_values(["encoder", "layer", "head"]).reset_index(drop=True)
summary.to_csv(OUT_DIR / "test_summary_with_bird_ci.csv", index=False)

# ---------- reproduction check against the archived final-layer results ----------
ARCHIVED_VAL = {  # all_validation_summaries_48.csv, budget 120, final layer
    ("HuBERT", "logreg"): 0.628593, ("Wav2Vec2", "logreg"): 0.471453,
    ("BirdAVES-base", "logreg"): 0.732928, ("BirdAVES-large", "logreg"): 0.743884,
    ("HuBERT", "mlp"): 0.638158, ("Wav2Vec2", "mlp"): 0.473870,
    ("BirdAVES-base", "mlp"): 0.729246, ("BirdAVES-large", "mlp"): 0.740119,
    ("HuBERT", "xgboost"): 0.607557, ("Wav2Vec2", "xgboost"): 0.434134,
    ("BirdAVES-base", "xgboost"): 0.712739, ("BirdAVES-large", "xgboost"): 0.704647,
    ("HuBERT", "bilstm"): 0.652891, ("Wav2Vec2", "bilstm"): 0.497843,
    ("BirdAVES-base", "bilstm"): 0.748544, ("BirdAVES-large", "bilstm"): 0.754233}
ARCHIVED_TEST_BILSTM = {"HuBERT": 0.555176, "Wav2Vec2": 0.402257,
                        "BirdAVES-base": 0.714298, "BirdAVES-large": 0.658493}
fin = summary[summary.is_final_layer]
chk = []
for _, r in fin.iterrows():
    a = ARCHIVED_VAL[(r.encoder, r["head"])]
    tol = 0.03 if r["head"] == "bilstm" else 0.01   # GPU kernels make the BiLSTM slightly non-deterministic
    chk.append(dict(encoder=r.encoder, head=r["head"], split="validation", archived=a, rerun=r.val_macro_f1,
                    diff=r.val_macro_f1 - a, status="REPRODUCED" if abs(r.val_macro_f1 - a) <= tol else "CHECK"))
    if r["head"] == "bilstm":
        a = ARCHIVED_TEST_BILSTM[r.encoder]
        chk.append(dict(encoder=r.encoder, head="bilstm", split="test", archived=a, rerun=r.test_macro_f1,
                        diff=r.test_macro_f1 - a, status="REPRODUCED" if abs(r.test_macro_f1 - a) <= 0.03 else "CHECK"))
chk = pd.DataFrame(chk)
chk.to_csv(OUT_DIR / "reproduction_check.csv", index=False)

pd.set_option("display.width", 200)
show = ["encoder", "layer", "n_layers", "head", "val_macro_f1", "test_macro_f1", "bird_boot_lo", "bird_boot_hi",
        "lobo_min", "lobo_max"] + [c for c in summary.columns if c.startswith("minus_mfcc_gboost")]
print("\nTest macro-F1 with bird-bootstrap 95% intervals")
print(summary[show].round(3).to_string(index=False))
print("\nReproduction of archived final-layer numbers")
print(chk.round(4).to_string(index=False))

## 9. Figure and results zip

In [ ]:
# ============================================================
# Cell 9. Layer-sweep figure and the results zip to send back
# ============================================================
import matplotlib
import matplotlib.pyplot as plt

COL = {"BirdAVES-base": "#2a78d6", "BirdAVES-large": "#eb6834", "HuBERT": "#1baf7a", "Wav2Vec2": "#eda100"}
fig, ax = plt.subplots(figsize=(6.4, 3.8))
for name in ENCODER_NAMES:
    s = sweep_summary[sweep_summary.encoder == name].sort_values("layer")
    ax.fill_between(s.relative_depth, s.val_macro_f1 - s.val_macro_f1_sd, s.val_macro_f1 + s.val_macro_f1_sd,
                    color=COL[name], alpha=0.15, lw=0)
    ax.plot(s.relative_depth, s.val_macro_f1, color=COL[name], lw=1.8, marker="o", ms=3, label=name)
    b = SELECTED[name]["best_layer"]
    ax.scatter([b / SELECTED[name]["final_layer"]], [SELECTED[name]["best_val_macro_f1"]], s=60,
               facecolor="white", edgecolor=COL[name], lw=1.8, zorder=5)
ax.set_xlabel("Relative layer depth (layer / number of layers)")
ax.set_ylabel("Validation macro-F1")
ax.set_xlim(0, 1.02)
ax.grid(alpha=0.25, lw=0.6)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=8, loc="lower right")
fig.tight_layout()
fig.savefig(OUT_DIR / "figures" / "layer_sweep.pdf")
fig.savefig(OUT_DIR / "figures" / "layer_sweep.png", dpi=220)
plt.show()

env = dict(python=platform.python_version(), torch=torch.__version__, sklearn=sklearn.__version__,
           xgboost=xgboost.__version__, numpy=np.__version__, pandas=pd.__version__,
           device=str(DEVICE), gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
           seeds=SEEDS, budget=BUDGET, c_grid=C_GRID, n_boot=N_BOOT,
           finished=time.strftime("%Y-%m-%d %H:%M:%S"))
(OUT_DIR / "run_environment.json").write_text(json.dumps(env, indent=2))

zip_base = WORK_ROOT / "zf_layer_test_results"
shutil.make_archive(str(zip_base), "zip", root_dir=OUT_DIR)   # the feature cache is kept outside OUT_DIR
size = (zip_base.with_suffix(".zip")).stat().st_size / 1e6
print(f"Send this file back: {zip_base.with_suffix('.zip')}  ({size:.1f} MB)")
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(OUT_DIR))